# 0. Install and Import Dependencies

In [1]:
!pip install mediapipe opencv-python

In [2]:
import cv2
import mediapipe as mp
import numpy as np
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

# 1. Make Detections

In [3]:
video_path = "../data/reference_videos/pushup_example.mp4"
cap = cv2.VideoCapture(video_path)

# Pose instance
with mp_pose.Pose(min_detection_confidence=0.5,
                  min_tracking_confidence=0.5) as pose:
    
    frame_idx = 0
    success_frames = 0
    failed_frames = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or frame is None:
            print("Video end")
            break

        # Recolor image to RGB because of Mediapipe to OpenCV
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = pose.process(image)

        # Results counting
        if results.pose_landmarks:
            success_frames += 1
        else:
            failed_frames += 1

        frame_idx += 1

    cap.release()
    cv2.destroyAllWindows()

    print(f"⚠️ Failed frames: {failed_frames}")

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1762603685.934982   16918 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1762603686.037523   16919 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1762603686.232605   16919 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


Video end
⚠️ Failed frames: 0


# 2. Determining Joints

In [4]:
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

In [5]:
def calculate_angle(a, b, c):
    a = np.array(a)  # shoulder
    b = np.array(b)  # elbow
    c = np.array(c)  # wrist

    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0]) 
    angle = np.abs(radians * 180.0 / np.pi)

    if angle > 180.0:
        angle = 360 - angle
        
    return angle

In [6]:
rep_count = 0
stage = None  # up or down
ANGLE_TOLERANCE = 20 

In [7]:
cap = cv2.VideoCapture("../data/reference_videos/pushup_example.mp4")

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Recolor image to RGB because of Mediapipe to OpenCV
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = pose.process(image)
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        try:
            landmarks = results.pose_landmarks.landmark # 33 mediapipe points

            # left body side
            l_shoulder = [landmarks[11].x, landmarks[11].y]
            l_elbow = [landmarks[13].x, landmarks[13].y]
            l_wrist = [landmarks[15].x, landmarks[15].y]
            l_angle = calculate_angle(l_shoulder, l_elbow, l_wrist)
            l_y = landmarks[11].y

            # right body side
            r_shoulder = [landmarks[12].x, landmarks[12].y]
            r_elbow = [landmarks[14].x, landmarks[14].y]
            r_wrist = [landmarks[16].x, landmarks[16].y]
            r_angle = calculate_angle(r_shoulder, r_elbow, r_wrist)
            r_y = landmarks[12].y

            # Pick side
            angle = min(l_angle, r_angle)
            shoulder_y = min(l_y, r_y)

            # Define thresholds with % tolerance
            is_down = angle < 70 * (1 + ANGLE_TOLERANCE / 100) and shoulder_y > 0.4
            is_up = angle > 160 * (1 - ANGLE_TOLERANCE / 100) and shoulder_y < 0.3

            if is_down:
                stage = "down"
            if is_up and stage == "down":
                rep_count += 1
                stage = "up"
                print(f"Rep counted! Total reps: {rep_count}")

        except:
            pass

        # Draw landmarks
        mp_drawing.draw_landmarks(
            image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=2),
            mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
        )

        # Visualization
        cv2.imshow('Push-up Rep Counter', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

W0000 00:00:1762603707.061929   16947 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1762603707.111728   16947 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Rep counted! Total reps: 1
Rep counted! Total reps: 2


# 3. Calculate Angles

In [19]:
import cv2
import mediapipe as mp
import numpy as np

# =============================================================
# 🧩 --- CONFIGURATION CONSTANTS ---
# =============================================================

ANGLE_TOLERANCE = 20                # % ανοχή γωνίας
CALIBRATION_SECONDS = 3             # Χρόνος που πρέπει να μείνει σταθερός ο χρήστης
TEXT_COLOR = (0, 0, 0)              # Μαύρο text για καλύτερη ορατότητα
ACCENT_COLOR = (0, 255, 127)        # Πράσινο για επιτυχία
WARNING_COLOR = (0, 165, 255)       # Πορτοκαλί για "down" φάση
PANEL_COLOR = (35, 35, 35)          # Background panel (αν ενεργοποιηθεί)
BORDER_COLOR = (80, 80, 80)

# =============================================================
# 🧠 --- HELPER FUNCTION ---
# =============================================================

def calculate_angle(a, b, c):
    """Υπολογίζει τη γωνία μεταξύ τριών σημείων (ώμος–αγκώνας–καρπός)."""
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    if angle > 180.0:
        angle = 360 - angle
    return angle

# =============================================================
# 🎥 --- VIDEO CAPTURE SETUP ---
# =============================================================

cap = cv2.VideoCapture(0)  # 0 = webcam. Μπορείς να βάλεις και path σε video file.
fps = cap.get(cv2.CAP_PROP_FPS)
if fps == 0:  # fallback σε περίπτωση που η κάμερα δεν επιστρέφει FPS
    fps = 30
# --- Smart Alignment Timing ---
ALIGN_FRAMES_REQUIRED = int(fps * 3)    # Μέγιστος χρόνος calibration (~3s)
MIN_ALIGN_FRAMES = int(fps * 0.7)       # Ελάχιστος χρόνος σταθερότητας (~0.7s)

mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

# =============================================================
# ⚙️ --- STATE VARIABLES ---
# =============================================================

rep_count = 0
stage = None  # "down" ή "up"
system_stage = "waiting"  # waiting → aligning → ready → counting
align_counter = 0
baseline_shoulder_y = None

# =============================================================
# 🧍 --- MAIN LOOP ---
# =============================================================

with mp_pose.Pose(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    model_complexity=0
) as pose:
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = pose.process(image)
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # =============================================================
        # ✅ EXPLICIT LANDMARK CHECK (χωρίς try/except)
        # =============================================================
        if not results.pose_landmarks:
            cv2.putText(image, "Searching for user...", (90, 150),
                        cv2.FONT_HERSHEY_DUPLEX, 1, (50, 50, 255), 2, cv2.LINE_AA)
            cv2.imshow('Push-up Counter', image)
            key = cv2.waitKey(10)
            if key == ord('q') or cv2.getWindowProperty('Push-up Counter', cv2.WND_PROP_VISIBLE) < 1:
                break
            continue

        # =============================================================
        # 📍 READ LANDMARKS
        # =============================================================
        landmarks = results.pose_landmarks.landmark

        # Left side
        l_shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x,
                      landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
        l_elbow = [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x,
                   landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y]
        l_wrist = [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x,
                   landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y]
        l_angle = calculate_angle(l_shoulder, l_elbow, l_wrist)
        l_y = landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y

        # Right side
        r_shoulder = [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x,
                      landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y]
        r_elbow = [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x,
                   landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y]
        r_wrist = [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x,
                   landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y]
        r_angle = calculate_angle(r_shoulder, r_elbow, r_wrist)
        r_y = landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y

        # --- Επιλογή μικρότερης γωνίας (πιο λυγισμένη πλευρά) ---
        angle = min(l_angle, r_angle)
        shoulder_y = min(l_y, r_y)
        h, w = image.shape[:2]
        shoulder_px = (int(l_shoulder[0] * w), int(l_shoulder[1] * h))

        # =============================================================
        # 🧠 SMART START DETECTION + PERSONAL CALIBRATION + REP LOGIC
        # =============================================================

        if system_stage == "waiting":
            cv2.putText(image, "Searching for user...", (90, 150),
                        cv2.FONT_HERSHEY_DUPLEX, 1, (50, 50, 255), 2, cv2.LINE_AA)
            if angle > 120:
                system_stage = "aligning"
                align_counter = 0

        elif system_stage == "aligning":
            cv2.putText(image, "Aligning posture...", (120, 150),
                        cv2.FONT_HERSHEY_DUPLEX, 1, (0, 255, 255), 2, cv2.LINE_AA)

            # Αν ο χρήστης έχει πάρει ευθεία θέση push-up (angle > 120)
            if angle > 120:
                align_counter += 1
            else:
                align_counter = 0

            if align_counter > MIN_ALIGN_FRAMES or align_counter > ALIGN_FRAMES_REQUIRED:
                baseline_shoulder_y = shoulder_y
                system_stage = "ready"

        elif system_stage == "ready":
            cv2.putText(image, "Are you ready to start Push-ups?", (60, 150),
                        cv2.FONT_HERSHEY_DUPLEX, 1, (0, 255, 0), 3, cv2.LINE_AA)
            if angle < 120:
                system_stage = "counting"

        elif system_stage == "counting":
            shoulder_shift = shoulder_y - baseline_shoulder_y
            down_threshold = 90 * (1 + ANGLE_TOLERANCE / 100)
            up_threshold = 140 * (1 - ANGLE_TOLERANCE / 100)

            if angle < down_threshold and shoulder_shift > 0.05:
                stage = "down"
            if angle > up_threshold and shoulder_shift < 0.02 and stage == "down":
                rep_count += 1
                stage = "up"

            # Dynamic color feedback
            color = WARNING_COLOR if stage == "down" else ACCENT_COLOR
            cv2.putText(image, f"Counting... Reps: {rep_count}", (60, 150),
                        cv2.FONT_HERSHEY_DUPLEX, 1, color, 2, cv2.LINE_AA)

        # =============================================================
        # 📊 VISUALIZATION (ANGLES + INFO)
        # =============================================================
        cv2.putText(image, f'{int(angle)}°', shoulder_px,
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, TEXT_COLOR, 2, cv2.LINE_AA)
        cv2.putText(image, f'ShY: {shoulder_y:.2f}', 
                    (shoulder_px[0], shoulder_px[1] + 25), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2, cv2.LINE_AA)

        # =============================================================
        # 🧾 PANEL OVERLAY (REPS / STAGE)
        # =============================================================
        cv2.putText(image, 'REPS', (25, 45),
                    cv2.FONT_HERSHEY_DUPLEX, 0.7, TEXT_COLOR, 2, cv2.LINE_AA)
        cv2.putText(image, f'{rep_count}', (150, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.8, ACCENT_COLOR, 3, cv2.LINE_AA)

        cv2.putText(image, 'STAGE', (25, 90),
                    cv2.FONT_HERSHEY_DUPLEX, 0.7, TEXT_COLOR, 2, cv2.LINE_AA)

        if stage == "down":
            stage_color = WARNING_COLOR
        elif stage == "up":
            stage_color = ACCENT_COLOR
        else:
            stage_color = (200, 200, 200)

        cv2.putText(image, f'{stage if stage else "-"}'.upper(),
                    (150, 95), cv2.FONT_HERSHEY_SIMPLEX, 1.2, stage_color, 3, cv2.LINE_AA)

        # =============================================================
        # 🦴 DRAW SKELETON
        # =============================================================
        mp_drawing.draw_landmarks(
            image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(245, 117, 66), thickness=2, circle_radius=2),
            mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=2)
        )

        # =============================================================
        # 🪟 DISPLAY & EXIT HANDLING
        # =============================================================
        cv2.imshow('Push-up Counter', image)
        key = cv2.waitKey(10)
        if key == ord('q') or cv2.getWindowProperty('Push-up Counter', cv2.WND_PROP_VISIBLE) < 1:
            break

cap.release()
cv2.destroyAllWindows()


W0000 00:00:1762604833.433249   17369 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1762604833.475916   17373 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
